# Tutorial 4: VQE for Portfolio Optimization

This notebook solves the portfolio QUBO using the Variational Quantum Eigensolver (VQE)
with a hardware-efficient ansatz. We compare VQE against QAOA on the same problem.

**Reference**: Barkoutsos et al. (2020) -- CVaR-VQE for combinatorial optimization.

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Problem Setup

In [ ]:
from qufin.portfolio.qubo import PortfolioQUBO
from qufin.backends.qiskit_backend import QiskitAerBackend

mu = np.array([0.12, 0.10, 0.07, 0.03, 0.15])
cov = np.array([
    [0.040, 0.006, 0.002, 0.000, 0.010],
    [0.006, 0.030, 0.004, 0.001, 0.008],
    [0.002, 0.004, 0.020, 0.002, 0.003],
    [0.000, 0.001, 0.002, 0.010, 0.001],
    [0.010, 0.008, 0.003, 0.001, 0.050],
])

qubo = PortfolioQUBO(mu=mu, cov=cov, gamma=0.5, cardinality=2)
backend = QiskitAerBackend(method="automatic", seed=42)  # shots passed at run()

print(f"Problem: {qubo.n_assets} assets, select {qubo.cardinality}")

## 2. VQE with Hardware-Efficient Ansatz

VQE uses a parameterized circuit (ansatz) with alternating layers of
single-qubit rotations and CNOT entangling gates.

In [ ]:
from qufin.portfolio.optimizers.vqe import VQEPortfolio, VQEConfig

config = VQEConfig(
    n_layers=3,          # Ansatz depth
    optimizer="COBYLA",
    maxiter=200,
    shots=4096,
    seed=42,
)

solver = VQEPortfolio(qubo, config, backend)
result = solver.run()

print(f"VQE result:")
print(f"  Best bitstring: {result.best_bitstring}")
print(f"  Objective:      {result.best_objective:.6f}")
print(f"  Feasible:       {result.feasible}")

## 3. Effect of Ansatz Depth

In [ ]:
for n_layers in [1, 2, 3, 4, 5]:
    cfg = VQEConfig(
        n_layers=n_layers, optimizer="COBYLA",
        maxiter=200, shots=4096, seed=42,
    )
    s = VQEPortfolio(qubo, cfg, backend)
    r = s.run()
    print(f"  layers={n_layers}: obj={r.best_objective:.6f}  bits={r.best_bitstring}  feasible={r.feasible}")

## 4. VQE vs QAOA Comparison

In [ ]:
from qufin.portfolio.optimizers.qaoa import QAOAPortfolio, QAOAConfig

qaoa_cfg = QAOAConfig(
    p=2, mixer="xy_ring", cardinality=2,
    optimizer="COBYLA", maxiter=200, shots=4096, seed=42,
)
qaoa_solver = QAOAPortfolio(qubo, qaoa_cfg, backend)
qaoa_result = qaoa_solver.run()

print(f"QAOA (p=2, XY-ring): obj={qaoa_result.best_objective:.6f}  bits={qaoa_result.best_bitstring}")
print(f"VQE  (3 layers):     obj={result.best_objective:.6f}  bits={result.best_bitstring}")

## 5. Warm-Start VQE

Initialize VQE parameters from a continuous relaxation solution.

In [ ]:
from qufin.portfolio.optimizers.warm_start import warm_start_vqe

# warm_start_vqe solves the continuous relaxation and returns initial ansatz
# parameters (n_params = rotation angles) biased toward the relaxed solution.
n_params = qubo.n_qubits * 3  # rotation angles for a 3-layer ansatz
ws_result = warm_start_vqe(qubo, n_params=n_params, seed=42)

print("Warm-start VQE (from continuous relaxation):")
print(f"  Rounded bitstring: {ws_result.rounded_bitstring}")
print(f"  Relaxed objective: {ws_result.relaxed_objective:.6f}")
print(f"  Initial params:    {np.round(ws_result.initial_params[:5], 4)} ...")

## Summary

In this tutorial we covered:
- VQE with hardware-efficient ansatz for portfolio QUBO
- Effect of ansatz depth on solution quality
- Head-to-head VQE vs. QAOA comparison
- Warm-start initialization from continuous relaxation

**Next**: Tutorial 05 transitions from Black-Scholes to quantum amplitude estimation for option pricing.